# test4_v2 — `test4.ipynb` 수정본

원본 `C:\t_project_1_1\test4.ipynb`은 원본 CSV 전체(260,601행)를 읽었습니다. 여기에는 팀 `final_holdout` 52,121행의 정답도 포함됩니다.
팀 규칙상 홀드아웃의 입력-정답 관계는 탐색하지 않으므로, 이 복사본은 **팀 development 208,480행만** 읽습니다.

분석 코드는 원본과 같고, 바뀐 것은 데이터 경로와 저장 경로뿐입니다. 그래서 건수와 비율 숫자는 원본과 다르게 나옵니다.

In [ ]:
# [v2] 팀 development 데이터만 사용 (final_holdout 제외)
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "preprocess.py").exists())
DEV_CSV = REPO / "data" / "development.csv"   # 원본 CSV에서 팀 홀드아웃 52,121행을 뺀 208,480행
OUT_DIR = REPO / "outputs" / "t_project_v2" / "test4"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("데이터:", DEV_CSV)
print("저장 폴더:", OUT_DIR)

데이터: C:\Users\jhj31\project_1\earthquake\data\development.csv
저장 폴더: C:\Users\jhj31\project_1\earthquake\outputs\t_project_v2\test4


In [2]:

import pandas as pd
import matplotlib.pyplot as plt

# 1. CSV 파일 불러오기
# VS Code에서 실제 사용하는 파일명으로 변경하세요.
df_original = pd.read_csv(DEV_CSV)  # [v2] 팀 development

# 2. 원본과 별도로 분석용 복사본 생성
df_eda = df_original.copy(deep=True)

# 3. 데이터 구조 확인
print("원본 데이터:", df_original.shape)
print("분석 데이터:", df_eda.shape)

# 4. 복사본 내용이 원본과 동일한지 확인
print("내용 일치:", df_original.equals(df_eda))

# 5. 처음 5행 확인
print(df_eda.head())


원본 데이터: (208480, 40)
분석 데이터: (208480, 40)
내용 일치: True
   building_id  geo_level_1_id  geo_level_2_id  geo_level_3_id  \
0            4              30             266            1224   
1            8              17             409           12182   
2           12              17             716            7056   
3           16               4             651             105   
4           25              26            1132            6645   

   count_floors_pre_eq  age  area_percentage  height_percentage  \
0                    1   25                5                  2   
1                    2    0               13                  7   
2                    2    5               12                  6   
3                    2   80                5                  4   
4                    2    0                6                  6   

  land_surface_condition foundation_type  ... has_secondary_use_hotel  \
0                      t               r  ...                       0   


In [3]:

# 전체 컬럼 진단표 생성
column_diagnosis = pd.DataFrame({
    "자료형": df_eda.dtypes.astype(str),
    "고유값수": df_eda.nunique(),
    "결측건수": df_eda.isna().sum(),
    "결측비율(%)": (
        df_eda.isna().mean() * 100
    ).round(2)
})

# 컬럼명을 일반 열로 변환
column_diagnosis = (
    column_diagnosis
    .reset_index()
    .rename(columns={"index": "컬럼명"})
)

# 40개 컬럼 모두 출력
print(column_diagnosis.to_string(index=False))

# 중복 진단
print("전체 행 중복:", df_eda.duplicated().sum())
print("ID 중복:", df_eda["building_id"].duplicated().sum())

print(
    "ID 제외 중복:",
    df_eda.drop(columns="building_id").duplicated().sum()
)


                                   컬럼명   자료형   고유값수  결측건수  결측비율(%)
                           building_id int64 208480     0      0.0
                        geo_level_1_id int64     31     0      0.0
                        geo_level_2_id int64   1408     0      0.0
                        geo_level_3_id int64  11353     0      0.0
                   count_floors_pre_eq int64      9     0      0.0
                                   age int64     41     0      0.0
                       area_percentage int64     83     0      0.0
                     height_percentage int64     26     0      0.0
                land_surface_condition   str      3     0      0.0
                       foundation_type   str      5     0      0.0
                             roof_type   str      3     0      0.0
                     ground_floor_type   str      5     0      0.0
                      other_floor_type   str      4     0      0.0
                              position   str      4     0     

전체 행 중복: 0
ID 중복: 0


ID 제외 중복: 8252


In [4]:

# 피해등급별 건수 집계
damage_summary = (
    df_eda["damage_grade"]
    .value_counts()
    .sort_index()
    .rename_axis("damage_grade")
    .reset_index(name="건수")
)

# 전체 건축물 대비 비율 계산
damage_summary["비율(%)"] = (
    damage_summary["건수"]
    / len(df_eda)
    * 100
).round(2)

print(damage_summary)


   damage_grade      건수  비율(%)
0             1   20099   9.64
1             2  118607  56.89
2             3   69774  33.47


In [5]:

# 1. 지역별 피해등급 교차표
region_summary = pd.crosstab(
    df_eda["geo_level_1_id"],
    df_eda["damage_grade"]
)

# 1, 2, 3등급 컬럼을 항상 유지
region_summary = region_summary.reindex(
    columns=[1, 2, 3],
    fill_value=0
)

# 2. 컬럼명 변경
region_summary = region_summary.rename(
    columns={
        1: "1등급_건수",
        2: "2등급_건수",
        3: "3등급_건수"
    }
)

region_summary.columns.name = None

# 3. 지역별 전체 건축물 수
region_summary["전체_건축물"] = (
    region_summary[
        ["1등급_건수", "2등급_건수", "3등급_건수"]
    ].sum(axis=1)
)

# 4. 2등급 비율
region_summary["2등급_비율(%)"] = (
    region_summary["2등급_건수"]
    / region_summary["전체_건축물"]
    * 100
).round(2)

# 5. 3등급 비율
region_summary["3등급_비율(%)"] = (
    region_summary["3등급_건수"]
    / region_summary["전체_건축물"]
    * 100
).round(2)

# 6. 네 가지 순위 각각 확인
ranking_columns = {
    "2등급 건수 순위": "2등급_건수",
    "3등급 건수 순위": "3등급_건수",
    "2등급 비율 순위": "2등급_비율(%)",
    "3등급 비율 순위": "3등급_비율(%)"
}

for title, column in ranking_columns.items():
    print(f"\n===== {title} =====")

    ranking = region_summary.sort_values(
        by=column,
        ascending=False
    )

    print(ranking.head(10).to_string())

# 7. 전체 건수 검증
assert region_summary["전체_건축물"].sum() == len(df_eda)

print("\n지역별 집계 검증 완료")



===== 2등급 건수 순위 =====
                1등급_건수  2등급_건수  3등급_건수  전체_건축물  2등급_비율(%)  3등급_비율(%)
geo_level_1_id                                                      
6                 1666   12984    4823   19473      66.68      24.77
26                6419   10087    1532   18038      55.92       8.49
10                 974    9706    7062   17742      54.71      39.80
20                2670    9467    1649   13786      68.67      11.96
7                  853    8975    5394   15222      58.96      35.44
4                  436    8894    2275   11605      76.64      19.60
8                  518    6833    7980   15331      44.57      52.05
13                1540    5084    1130    7754      65.57      14.57
27                 382    4788    4845   10015      47.81      48.38
21                 256    4711    6897   11864      39.71      58.13

===== 3등급 건수 순위 =====
                1등급_건수  2등급_건수  3등급_건수  전체_건축물  2등급_비율(%)  3등급_비율(%)
geo_level_1_id                                           

In [6]:
 #1. 지역별 피해등급 교차표

# 대지역 + 중지역별 피해등급 교차표
region_summary = pd.crosstab(
    index=[
        df_eda["geo_level_1_id"],
        df_eda["geo_level_2_id"]
    ],
    columns=df_eda["damage_grade"]
)

# 피해등급 1, 2, 3 순서로 정리
region_summary = region_summary.reindex(
    columns=[1, 2, 3],
    fill_value=0
)

# 컬럼 이름 변경
region_summary.columns = [
    "1등급_건수",
    "2등급_건수",
    "3등급_건수"
]

display(region_summary.head(20))
region_summary.columns.name = None

# 3. 지역별 전체 건축물 수
region_summary["전체_건축물"] = (
    region_summary[
        ["1등급_건수", "2등급_건수", "3등급_건수"]
    ].sum(axis=1)
)

# 4. 2등급 비율
region_summary["2등급_비율(%)"] = (
    region_summary["2등급_건수"]
    / region_summary["전체_건축물"]
    * 100
).round(2)

# 5. 3등급 비율
region_summary["3등급_비율(%)"] = (
    region_summary["3등급_건수"]
    / region_summary["전체_건축물"]
    * 100
).round(2)

# 6. 네 가지 순위 각각 확인
ranking_columns = {
    "2등급 건수 순위": "2등급_건수",
    "3등급 건수 순위": "3등급_건수",
    "2등급 비율 순위": "2등급_비율(%)",
    "3등급 비율 순위": "3등급_비율(%)"
}

for title, column in ranking_columns.items():
    print(f"\n===== {title} =====")

    ranking = region_summary.sort_values(
        by=column,
        ascending=False
    )

    print(ranking.head(10).to_string())

# 7. 전체 건수 검증
assert region_summary["전체_건축물"].sum() == len(df_eda)

print("\n지역별 집계 검증 완료")


1등급_건수  2등급_건수  3등급_건수
geo_level_1_id geo_level_2_id                        
0              62                   0      28       5
               69                  10      43       6
               81                   2      42       5
               114                  0      21       2
               146                  1      52      15
               149                  1      18       3
               152                  2      48      10
               153                  1       3       3
               163                  0      38       5
               200                  0       5       3
               245                  2      61      10
               252                  1      65       7
               279                  0      18       6
               291                  2      14       5
               332                  0       5       4
               357                  5      41       2
               462                  3      20       4
               536                  5      73       9
               540                  0      44       9
               549                  0      67       7


===== 2등급 건수 순위 =====
                               1등급_건수  2등급_건수  3등급_건수  전체_건축물  2등급_비율(%)  3등급_비율(%)
geo_level_1_id geo_level_2_id                                                      
26             39                1631    1471     127    3229      45.56       3.93
20             158                426    1367     215    2008      68.08      10.71
3              1387                62    1156     429    1647      70.19      26.05
27             181                 74    1059     557    1690      62.66      32.96
6              673                309     966      81    1356      71.24       5.97
7              157                346     941     255    1542      61.02      16.54
11             883                 57     853     414    1324      64.43      31.27
25             889                 63     730     103     896      81.47      11.50
8              463                257     595     530    1382      43.05      38.35
3              720                 18     584     474